#1&rpar; 📈 Dask

##1.1. What is Dask?
It is an open source, distributed, parellel processing framework for Big Data. It is very use and set up as it essentially scales exisiting python.


##1.2. Dask vs Spark - Tradeoffs


1. Dask's SQL engine is still premature. Unlike Spark, we cannot manipulate our data with SQL queries.

2. Dask is not fault tolerant like Spark, which offers fault tolenrance out of the box through lineage.

3. Spark offers dedicated, production-ready libraries like GraphX for graph processing and a more cohesive MLLib for machine learning. Dask relies on integrations with external libraries like Scikit-Learn or XGBoost, which can feel less integrated.

## 1.3. What are the 3 main data structures in Dask, and what do they scale?
 1. **DASK ARRAY** - It scales the NumPy ndarray.
 2. **DASK DATAFRAME** - It scales to process large data by parallelizating Pandas Dataframes.
 3. **DASK BAG** - Scales standard Python lists, dictionaries and iterables, effectively handling unstructured or semi structured data. (similar to pyspark RDD)


## 1.4 What are Dask Delayed and Dask Futures?

These are the two low-level collection APIS for DASK computations:



*   Dask Delayed - it decorates our function so that they operate lazily. Rather than executing the function immediately, it will defer execution, placing the function and its argument into a task graph.
*   Dask Futures - An interface for immediate execution on a Dask cluster. When you submit a function using Futures, it begins executing immediately in the background and returns  a "future" object that points to the result.



##1.5. What is Dask-ML and what does it scale? Why is using Dask-ML advantageous?

Dask ML provides scalable machine learning in python. It scales scikit-learn for distributed parellel computing.


It is advantageous because it allows to train models on datasets that exceed RAM limits and distribute hyperparameter tuning across clusters without needing to rewrite the code into a new ecosystem while preserving the familiarity of scikit learn syntax.


##1.6. Explain how Lazy Evaluation works in Dask.

Dask uses lazy evaluation, meaning computations are not executed immediately. Instead Dask builds a task graph representing the computation, and only performs the actual work when we explicitly trigger it with methods like .compute() or persist(). This allows dask to optimize and parallelize workflow efficiently before execution.  For example, operations on Dask arrays or Dataframes just build up the computation plan, and nothing is computed until you call .compute() to get the results in memory, or persist() to keep results distributed in memory for further use.

This approach enables it to optimize the computation, share intermediate results and avoid unnecessary work. It is especially useful for large datasets and it can minimize memory usage and execution time by computing what is needed only when it is needed.

#2&rpar; 👷🏿‍♀️🔨 Dask -  Set Up


In [ ]:
!pip install dask-ml -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.0/150.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.4 MB/s eta 0:00:00


Kaggle Dataset Link : https://www.kaggle.com/datasets/nilesh2042/airport-traffic-dataset/data

In [ ]:
import kagglehub
import os
import dask.dataframe as dd

path = kagglehub.dataset_download("nilesh2042/airport-traffic-dataset")

print("Path to dataset files:", path)

file_path = os.path.join(path, "airport_traffic_2025.csv")

df = dd.read_csv(file_path)
df.head()

Using Colab cache for faster access to the 'airport-traffic-dataset' dataset.
Path to dataset files: /kaggle/input/airport-traffic-dataset


,YEAR,MONTH_NUM,MONTH_MON,FLT_DATE,APT_ICAO,APT_NAME,STATE_NAME,FLT_DEP_1,FLT_ARR_1,FLT_TOT_1,FLT_DEP_IFR_2,FLT_ARR_IFR_2,FLT_TOT_IFR_2
0,2025,1,JAN,2025-01-01,LATI,Tirana,Albania,64,62,126,NaN,NaN,NaN
1,2025,1,JAN,2025-01-01,UDYZ,Yerevan,Armenia,57,54,111,NaN,NaN,NaN
2,2025,1,JAN,2025-01-01,LOWG,Graz,Austria,7,6,13,NaN,NaN,NaN
3,2025,1,JAN,2025-01-01,LOWI,Innsbruck,Austria,24,25,49,NaN,NaN,NaN
4,2025,1,JAN,2025-01-01,LOWK,Klagenfurt,Austria,2,0,2,NaN,NaN,NaN


# 3&rpar; 🧹 Data cleaning

The cleaning / preprocessing step we did was removing the columns [FLT_DEP_IFR_2, FLT_ARR_IFR_2 and FLT_TOT_IFR_2]  as there were 81892 missing values for each.

In [ ]:
cols_to_drop = ["FLT_TOT_IFR_2","FLT_ARR_IFR_2","FLT_DEP_IFR_2"]
df_clean = df.drop(columns=cols_to_drop)
df_clean.head()

,YEAR,MONTH_NUM,MONTH_MON,FLT_DATE,APT_ICAO,APT_NAME,STATE_NAME,FLT_DEP_1,FLT_ARR_1,FLT_TOT_1
0,2025,1,JAN,2025-01-01,LATI,Tirana,Albania,64,62,126
1,2025,1,JAN,2025-01-01,UDYZ,Yerevan,Armenia,57,54,111
2,2025,1,JAN,2025-01-01,LOWG,Graz,Austria,7,6,13
3,2025,1,JAN,2025-01-01,LOWI,Innsbruck,Austria,24,25,49
4,2025,1,JAN,2025-01-01,LOWK,Klagenfurt,Austria,2,0,2


# 4&rpar; 🤖  Machine learning

While Dask-ML interfaces beautifully with Scikit learn, fully distributed. native Random Forests for massive, out-of-the core datasets are not build natively into Dask in the exact same way they are in Spark MLlib. Dask-ML focuses on scalable machine learning and provides some ensemble methods, but RandomForestRegressor is not explicitly listed among its available estimators. Instead, Dask-ML often relies on integrations with libraries like Scikit-learn and XGBoost for tree-based models, and users can parallelize Scikit-learn's RandomForestRegressor using Dask's joblib backend or use XGBoost for distributed gradient boosting regression tasks. To stay as close to the Scikit-Learn Random Forest structure used previously, we will use Scikit-Learn's implementation backed by Dask's parallel joblib integration.

In [ ]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from dask_ml.preprocessing import Categorizer, OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from dask.distributed import Client

client = Client()

categorizer = Categorizer(columns=["STATE_NAME","APT_ICAO"])
df_clean = categorizer.fit_transform(df_clean)

encoder = OrdinalEncoder(columns=["STATE_NAME","APT_ICAO"])
df_clean = encoder.fit_transform(df_clean)

X_dask = df_clean[['MONTH_NUM','STATE_NAME',"APT_ICAO"]]
y_dask = df_clean['FLT_TOT_1']

X = X_dask.compute()
y = y_dask.compute()

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,shuffle=True)

rf_model = RandomForestRegressor(n_estimators=50, random_state = 42)

with joblib.parallel_backend('dask'):
  rf_model.fit(X_train,y_train)
  predictions = rf_model.predict(X_test)

mse = mean_squared_error(y_test,predictions)

print(f"Mean Squared error = {mse}")


INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:45337
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:36281'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:41207'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:33381 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:33381
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:39118
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:41299 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:41299
INFO:distributed.core:Starting established connection to tcp://127

Mean Squared error = 582.9373079352459


Chosen metric to model evaluation is mean squared error (MSE)
Result is 582.9

What this means?

Mean Squared error calculates the average of the squared differences between the model's predicted flight volumes and the actual flight volumes. By squaring the errors, MSE heavily penalizes predictions that are wildly off-target.

Success is measured by minimizing the MSE, a lower number implies that the model's predictions are closer to the actual values. However, because MSE is represented in "squared flights" (which isn't intuitive), it is often helpful to look at the Root Mean Squared Error (RMSE) by taking the square root of our MSE.

so, &radic;583 ~ 24

This means that on average, our Random Forest model's predictions are off by 25 flights per state/month combination. Considering that major airports habdle thousands of flights a month, being off by an average of only 24 indicated that the model has successfully identified the underlying seasonal and geographic patterns in trafiic data.

#5&rpar; 📊 Data pipelines

##5.1. What is a data pipeline?

A data pipeline is a method in which raw data is ingested from various data sources, transformed and then ported to a data store, such as a data lake or data warehouse, for analysis.

##5.2. Why do data scientists need to know about data pipelines?

A data pipeline automates the entire lifecycle of data, from continuous extraction and preprocessing to model training and deployment. It ensures that data workflows are modular, reproducible, testable and automated. It makes it easy to deploy models into production environments where fresh data needs to be procesed continuously.


##5.3. Step 4 as a Data pipeline

In [ ]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from dask_ml.preprocessing import Categorizer, OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from dask.distributed import Client

def ingest_data(filepath):
  return dd.read_csv(filepath)

def clean_data(df):
  df = df.dropna(subset=['MONTH_NUM','STATE_NAME',"APT_ICAO"])
  categorizer = Categorizer(columns=["STATE_NAME","APT_ICAO"])
  df = categorizer.fit_transform(df)

  encoder = OrdinalEncoder(columns=["STATE_NAME","APT_ICAO"])
  df = encoder.fit_transform(df)
  return df

def prepare_data(df):
  X_dask = df_clean[['MONTH_NUM','STATE_NAME',"APT_ICAO"]]
  y_dask = df_clean['FLT_TOT_1']

  X = X_dask.compute()
  y = y_dask.compute()
  return X, y

def train_and_evaluate(X,y):
  X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,shuffle=True)

  rf_model = RandomForestRegressor(n_estimators=50, random_state = 42)

  with joblib.parallel_backend('dask'):
    rf_model.fit(X_train,y_train)
    predictions = rf_model.predict(X_test)

  mse = mean_squared_error(y_test,predictions)
  print(f"Mean Squared error = {mse}")
  return mse

def run_pipeline(filepath):
  """Executes the pipeline stages in sequence."""
  print("🔨⛏️ Starting pipeline...")
  client = Client()

  print("1️⃣ Ingesting data...")
  raw_df = ingest_data(filepath)

  print("2️⃣ Cleaning data...")
  clean_df = clean_data(raw_df)

  print("3️⃣ Preprocessing the data...")
  X,y = prepare_data(clean_df)

  print("4️⃣ Training model and evaluationg...")
  mse = train_and_evaluate(X,y)

  print(" ✅ Pipeline completed successfully!")
run_pipeline(file_path)

/usr/local/lib/python3.12/dist-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 34159 instead
  warnings.warn(
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:38789
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:34159/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:35981'


🔨⛏️ Starting pipeline...


INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:45665'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:46673 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:46673
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:35562
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:44159 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:44159
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:35572
INFO:distributed.scheduler:Receive client connection: Client-56daf955-4695-11f1-9514-0242ac1c000c
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:35580


1️⃣ Ingesting data...
2️⃣ Cleaning data...
3️⃣ Preprocessing the data...
4️⃣ Training model and evaluationg...


INFO:distributed.scheduler:Receive client connection: Client-worker-5bc01418-4695-11f1-96a2-0242ac1c000c
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:35626
INFO:distributed.scheduler:Receive client connection: Client-worker-5bc15faa-4695-11f1-969f-0242ac1c000c
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:35630


Mean Squared error = 601.6766712715695
 ✅ Pipeline completed successfully!
